In [17]:
import pandas as pd

data=pd.read_csv('datasets/final_reviews_with_categories.csv')
data.columns

Index(['rating', 'title', 'text', 'images', 'asin', 'parent_asin', 'user_id',
       'timestamp', 'helpful_vote', 'verified_purchase', 'product_name',
       'topic_id', 'bertopic_category_name', 'average_rating', 'rating_number',
       'original_images'],
      dtype='object')

In [18]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv()) 
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [19]:
def get_balanced_priority_reviews(group, n_per_rating=10):

    all_selected = []
    
    for r in [1, 2, 3, 4, 5]:
        mask = group['rating'] == r
        bucket = group[mask]
        
        if bucket.empty:
            # Fallback: If a specific rating is missing, we just move on
            continue
            
        # PRIMARY SORT: Verified + Helpful
        # SECONDARY SORT: Helpful only (even if not verified)
        # TERTIARY: Just the text (ensures we get 'n' reviews if they exist)
        sorted_bucket = bucket.sort_values(
            by=['verified_purchase', 'helpful_vote'], 
            ascending=[False, False]
        ).head(n_per_rating)
        
        all_selected.append(sorted_bucket)
    
    if not all_selected:
        return "No reviews available for this product."
    print(len(all_selected), " selected")
 
    combined = pd.concat(all_selected)
    
    # Add a prefix to each review so the AI knows the rating/status
    review_strings = []
    for _, row in combined.iterrows():
        status = "Verified" if row['verified_purchase'] else "Unverified"
        review_strings.append(f"[{row['rating']}-Stars | {status}]: {row['text']}")

    combined = pd.concat(all_selected)
    
    num_reviews_found = len(combined)
    print(f"Total reviews being sent to AI: {num_reviews_found}")
    print("----")
        
    return " [NEXT REVIEW] ".join(review_strings)

In [ ]:
import openai

client = openai.OpenAI(api_key=OPENAI_API_KEY)

def summarize_product(name, reviews):
    """Task 2: Create a detailed summary for an individual product."""
    prompt = f"""Summarize the reviews for '{name}'. 
    The reviews provided are balanced across all rating levels (1-5 stars).
    Please provide:
    - Overall Sentiment
    - Top 3 Pros (what loyal fans love)
    - Top 3 Cons (common complaints from 1-2 star reviews)
    - Best For: (who should buy this?)
    
    REVIEWS:
    {reviews}"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=600
    )
    return response.choices[0].message.content

In [21]:
def generate_category_article(category, summaries):
    """Task 1: Generate the final 'Best of' recommendation article."""
    summaries_text = "\n\n---\n\n".join(summaries)
    prompt = f"""You are a professional consumer journalist. Write a 'Top 10 Best {category}' article.
    Use the following individual product summaries to build the guide.
    Include a 'Winner' for the category and a brief 'Buying Guide' section.
    
    SUMMARIES:
    {summaries_text}"""
    
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7
    )
    return response.choices[0].message.content

In [22]:
output_folder="summaries"
if not os.path.exists(output_folder):
    os.makedirs(output_folder)

In [23]:
product_meta = (
    data.groupby(['bertopic_category_name', 'product_name'], as_index=False)
    .agg(
        average_rating=('average_rating', 'first'),
        rating_number=('rating_number', 'first'),
        review_count=('text', 'count') 
    )
)

qualified_products = product_meta[
    (product_meta['average_rating'] >= 4.5) &
    (product_meta['rating_number'] >= 100) &
    (product_meta['review_count'] >= 50)
]


categories = qualified_products['bertopic_category_name'].unique()
print(f"Found {len(qualified_products)} qualifying products across {len(categories)} categories.")

for cat in categories:
    print(f"\nProcessing Category: {cat}")

    top_10 = (
        qualified_products[qualified_products['bertopic_category_name'] == cat]
        .sort_values(by='average_rating', ascending=False)
        .head(10)
    )

    if top_10.empty:
        print(f"  - No qualifying products for {cat}, skipping.")
        continue

    print(f"  - {len(top_10)} products selected.")
    product_summaries = []

    for _, row in top_10.iterrows():
        p_name = row['product_name']
        print(f"  - Collecting reviews: {p_name} ({row['rating_number']} ratings, avg {row['average_rating']:.2f}★)")

        product_reviews = data[data['product_name'] == p_name]
        sampled_text = get_balanced_priority_reviews(product_reviews)

        print(f"    Summarizing...")
        summary = summarize_product(p_name, sampled_text)
        product_summaries.append(f"PRODUCT: {p_name}\n{summary}")

    print(f"  - Generating final article for {cat}...")
    final_article = generate_category_article(cat, product_summaries)

    cat_filename = cat.replace(' ', '_')
    with open(os.path.join(output_folder, f"Summary_{cat_filename}.txt"), "w", encoding="utf-8") as f:
        f.write("\n\n".join(product_summaries))
    with open(os.path.join(output_folder, f"Article_{cat_filename}.txt"), "w", encoding="utf-8") as f:
        f.write(final_article)

print("\nDone! All summaries and articles have been generated.")

Found 489 qualifying products across 4 categories.

Processing Category: Hair Accessories And Braids
  - 10 products selected.
  - Collecting reviews: 4.5 Inch Hair Bows For Girls Grosgrain Ribbon Boutique Bow Clips Teens Toddlers Kids Set Of 30 (15 Colors x 2) (1085 ratings, avg 4.80★)
4  selected
Total reviews being sent to AI: 29
----
    Summarizing...
  - Collecting reviews: QtGirl Hair Bow Holder Organizer for Girls, Unicorn Bow Hanger Hair Clips Storage Hair Accessories Headband Storage for Girls Room (874 ratings, avg 4.80★)
5  selected
Total reviews being sent to AI: 18
----
    Summarizing...
  - Collecting reviews: Picoway 20 Pack Mouse Ears Solid Black and Red Bow Headband (11485 ratings, avg 4.80★)
5  selected
Total reviews being sent to AI: 49
----
    Summarizing...
  - Collecting reviews: NEWTGAN 20 PCS Mouse Ears for Birthday Party Theme Park Costume Play Celebration for Boys and Girls … (2231 ratings, avg 4.80★)
5  selected
Total reviews being sent to AI: 26
----
    